# Multi-Horizon Stock Market Forecasting System
## Part 2: PyTorch LSTM, GRU, Attention, Time-Series Transformer & Backtesting

In this notebook, we implement and train high-performance deep learning models in PyTorch:
1. **PyTorch LSTM & GRU**: Modern sequence architectures tailored for numeric time series.
2. **Attention-LSTM**: A Sequence-to-Sequence model with additive Bahdanau temporal attention. We will extract and visualize its attention weights to see what the model focus is.
3. **Time-Series Transformer**: A self-attention neural network utilizing positional encodings.
4. **Quantitative Backtesting Engine**: Testing model predictions in a simulated trading environment to compute Cumulative Returns, Annualized Sharpe Ratio, Max Drawdown, and Win Rates.

### 1. Setup & Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sys.path.append(os.path.abspath('..'))

from src.utils import logger, setup_plotting_style, plot_predictions, plot_backtest_results, plot_attention_weights
from src.models.lstm_gru import PyTorchTrainer
from src.models.attention import AttentionLSTMHelper
from src.models.transformer import TransformerHelper
from src.evaluation import calculate_forecasting_metrics, QuantitativeBacktester

setup_plotting_style()
print(f"CUDA available? {torch.cuda.is_available()}")

### 2. Prepare Feature Dataset & Train-Test Splits

In [ ]:
ticker = 'RELIANCE.NS'
features_path = os.path.join('..', 'data', 'processed', f"{ticker.replace('.NS', '')}_features.csv")

if not os.path.exists(features_path):
    raise FileNotFoundError("Please execute Notebook Part 1 first to generate and save features!")
    
df_features = pd.read_csv(features_path, index_col=0, parse_dates=True)

# Split: 80% train_val, 20% test
split_idx = int(len(df_features) * 0.8)
train_val_data = df_features.iloc[:split_idx]
test_data = df_features.iloc[split_idx:]

# Sub-split train_val into train and validation sets (90/10) for PyTorch early stopping validation
sub_split = int(len(train_val_data) * 0.9)
train_data = train_val_data.iloc[:sub_split]
val_data = train_val_data.iloc[sub_split:]

target_col = 'Close'
feature_cols = [c for c in df_features.columns if c not in ['Close', 'Adj Close', 'Returns', 'Log_Returns']]

print(f"Train samples:      {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples:       {len(test_data)}")
print(f"Features dimension: {len(feature_cols)}")

### 3. Initialize PyTorch Trainer
We configure our deep learning trainer using a 20-day lookback window to forecast a 1-day horizon.

In [ ]:
trainer = PyTorchTrainer(
    lookback=20,
    horizon=1,
    hidden_dim=64,
    num_layers=2,
    lr=0.001,
    batch_size=32,
    epochs=20,     # Low epochs for fast execution in notebook, increase as desired
    patience=5
)

### 4. Train LSTM Model

In [ ]:
lstm_weights = "../results/models/reliance_lstm.pth"
os.makedirs(os.path.dirname(lstm_weights), exist_ok=True)

trainer.fit(train_data, val_data, feature_cols, target_col, model_type='lstm', save_path=lstm_weights)
lstm_preds = trainer.predict(test_data, feature_cols)[:, 0]
print(f"LSTM predictions computed. Shape: {lstm_preds.shape}")

### 5. Train GRU Model

In [ ]:
gru_weights = "../results/models/reliance_gru.pth"
trainer.fit(train_data, val_data, feature_cols, target_col, model_type='gru', save_path=gru_weights)
gru_preds = trainer.predict(test_data, feature_cols)[:, 0]
print(f"GRU predictions computed. Shape: {gru_preds.shape}")

### 6. Train Attention-LSTM & Visualize Weights
The attention model enables the net to dynamically weight historical lookback steps.

In [ ]:
attn_helper = AttentionLSTMHelper(trainer)
attn_weights_path = "../results/models/reliance_attn_lstm.pth"

attn_helper.fit(train_data, val_data, feature_cols, target_col, save_path=attn_weights_path)
attn_preds = trainer.predict(test_data, feature_cols)[:, 0]

# Extract attention weights
mean_attention, _ = attn_helper.extract_attention_weights(test_data, feature_cols)
print(f"Attention weights extracted.")

Let's plot the average temporal attention weights across our 20-day lookback sequence to see what periods are heavily prioritized by our network.

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(x=[f"t-{20-i}" for i in range(20)], y=mean_attention, color='#7209B7')
plt.title("Mean Temporal Attention Distribution (Bahdanau)", fontweight='bold', fontsize=13)
plt.xlabel("Lookback Time-Steps")
plt.ylabel("Attention Weight")
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

### 7. Train Custom Time-Series Transformer

In [ ]:
trans_helper = TransformerHelper(trainer)
trans_weights_path = "../results/models/reliance_transformer.pth"

trans_helper.fit(train_data, val_data, feature_cols, target_col, save_path=trans_weights_path)
trans_preds = trainer.predict(test_data, feature_cols)[:, 0]
print(f"Transformer predictions computed.")

### 8. Compare Deep Forecasting Architectures

In [ ]:
# Set up targets: deep learning model requires lookback sliding alignment
offset = 20
aligned_dates = test_data.index[offset:]
aligned_actuals = test_data[target_col].values[offset:]

# Cut forecasts to match aligned dates size
aligned_lstm = lstm_preds
aligned_gru = gru_preds
aligned_attn = attn_preds
aligned_trans = trans_preds

lstm_met = calculate_forecasting_metrics(aligned_actuals, aligned_lstm, "LSTM")
gru_met = calculate_forecasting_metrics(aligned_actuals, aligned_gru, "GRU")
attn_met = calculate_forecasting_metrics(aligned_actuals, aligned_attn, "Attention-LSTM")
trans_met = calculate_forecasting_metrics(aligned_actuals, aligned_trans, "Transformer")

# Plot all deep learning forecasts
plt.figure(figsize=(14, 7))
plt.plot(aligned_dates, aligned_actuals, label='Actual Close', color='#2B2D42', linewidth=2.0)
plt.plot(aligned_dates, aligned_lstm, label='LSTM Forecast', color='#00B4D8', linestyle='--')
plt.plot(aligned_dates, aligned_gru, label='GRU Forecast', color='#7209B7', linestyle='--')
plt.plot(aligned_dates, aligned_attn, label='Attention-LSTM Forecast', color='#4CC9F0', linestyle='--')
plt.plot(aligned_dates, aligned_trans, label='Transformer Forecast', color='#38B000', linestyle='--')
plt.title(f"{ticker} - Neural Networks Forecasts Comparison on Test Data", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.show()

### 9. Strategy Backtesting: ARIMA vs Deep Learning vs Buy-and-Hold

In [ ]:
backtester = QuantitativeBacktester(initial_capital=100000.0)

stats_list = []
equity_curves = {}

# Buy and Hold
bh_stats, _, bh_curve = backtester.backtest(aligned_dates, aligned_actuals, aligned_actuals)
equity_curves['Buy & Hold'] = bh_curve

# LSTM
lstm_stats, lstm_curve, _ = backtester.backtest(aligned_dates, aligned_actuals, aligned_lstm)
stats_list.append({**lstm_stats, 'Model': 'LSTM'})
equity_curves['LSTM Strategy'] = lstm_curve

# GRU
gru_stats, gru_curve, _ = backtester.backtest(aligned_dates, aligned_actuals, aligned_gru)
stats_list.append({**gru_stats, 'Model': 'GRU'})
equity_curves['GRU Strategy'] = gru_curve

# Attention-LSTM
attn_stats, attn_curve, _ = backtester.backtest(aligned_dates, aligned_actuals, aligned_attn)
stats_list.append({**attn_stats, 'Model': 'Attention-LSTM'})
equity_curves['Attention-LSTM Strategy'] = attn_curve

# Transformer
trans_stats, trans_curve, _ = backtester.backtest(aligned_dates, aligned_actuals, aligned_trans)
stats_list.append({**trans_stats, 'Model': 'Transformer'})
equity_curves['Transformer Strategy'] = trans_curve

# Create backtest stats comparison table
df_backtest_comp = pd.DataFrame(stats_list).set_index('Model')
print(df_backtest_comp.drop(columns=['Strategy Type']))

Finally, let's plot the cumulative return path for all strategies relative to the Benchmark Buy & Hold strategy.

In [ ]:
plt.figure(figsize=(14, 7))
for name, curve in equity_curves.items():
    pct_ret = (curve - 100000.0) / 100000.0 * 100.0
    plt.plot(aligned_dates, pct_ret, label=name, linewidth=1.5 if 'Strategy' in name else 2.0)

plt.title(f"{ticker} - Strategy Equity Curves Comparison", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Cumulative Return (%)')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()